In [ ]:
load_ext jupyter_black

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from utils import extract_demographics
from scipy.stats import ttest_ind

In [ ]:
cat_cols = [
    "age",
    "gender",
    "education",
    "occupation",
    # "country", country is US for all included personas
    "race",
    "nationality",
    "relationship",
    "political",
    "income",
    "income_sat",
    "religion",
    # "conditions", nearly all personas have no medical conditions
    "culture_pride",
    "community_connected",
    "community_welcome",
    "race_fairness",
    "culture_skills",
    # "topic_label_persona",
]

num_cols = [
    # "care",
    # "money",
    # "equality",
    # "admiration",
    # "humility",
    # "spoil",
    # "risks",
    # "security",
    # "independence",
    # "leadership",
    # "loyalty",
    # "nature",
    # "success",
    # "customs",
    # "indulgence",
    # "novelty",
    # "government",
    # "proper",
    # "creativity",
    # "respect",
    # "short-term_goals_positive_social_impact",
    # "short-term_goals_start_business",
    # "short-term_goals_fulfilling_career",
    # "short-term_goals_creative_life",
    # "short-term_goals_personal_freedom",
    # "short-term_goals_family",
    # "short-term_goals_expert",
    # "long-term_goals_financial_independence",
    # "long-term_goals_positive_social_impact",
    # "long-term_goals_start_business",
    # "long-term_goals_fulfilling_career",
    # "long-term_goals_creative_life",
    # "long-term_goals_personal_freedom",
    # "long-term_goals_family",
    # "long-term_goals_expert",
    # "outgoing",
    # "dominant",
    # "energetic",
    # "quiet",
    # "follower",
    # "less_active",
    # "compassionate",
    # "respectful",
    # "unsympathetic",
    # "argumentative",
    # "rude",
    # "organized",
    # "careful",
    # "messy",
    # "careless",
    # "irresponsible",
    # "worries",
    # "tense",
    # "sensitive",
    # "relaxed",
    # "not_anxious",
    # "calm",
    # "original",
    # "art_beauty",
    # "curious",
    # "not_abstract",
    # "avoid_discussions",
    # "not_artistic",
    # "inherent_vulnerability",
    # "situational_vulnerability",
    # "pathogenic_vulnerability",
    "vulnerability_score",
]
p = 0.05 / len(num_cols)

In [ ]:
n = 25
main_models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3-8B",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
    # "Olmo-3-7B-Instruct",
]
mitigation_models = [
    "Llama-3.1-8B-Instruct",
    "Llama-3.1-8B-Instruct_mitigate",
    "dpo_model_Llama-3.1-8B-Instruct",
    "Qwen3-8B",
    "Qwen3-8B_mitigate",
    "dpo_model_Qwen3-8B",
]

df_personas = pd.read_pickle("data/personas_200_ch.gz")
df_vulnerability = pd.read_csv(
    "data/personas_demographics_vulnerability_final (1).csv"
)[
    [
        "uuid",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
]
df_personas = df_personas.merge(df_vulnerability, on="uuid")

In [ ]:
# df_persona_topic = pd.read_csv("topic_modeling/persona/document_topic_assignments.csv")
# df_persona_topic["persona_id"] = pd.read_csv("data/base_data_persona.csv")[
#     "persona_id"
# ].tolist()[:-1]

# df_persona_topic = df_persona_topic.drop(
#     columns=[
#         "persona_text",
#         "id",
#         "dominant_topic",
#         "topic_prob",
#         "low_confidence",
#         "topic_composition",
#     ]
# )
# df_persona_topic["topic_label"] = df_persona_topic["topic_label"].str.capitalize()
# df_persona_topic = df_persona_topic.rename(
#     columns={"topic_label": "topic_label_persona"}
# )

# df_prompt_topic = pd.read_csv("topic_modeling/prompt/document_topic_assignments.csv")
# df_prompt_topic["prompt_id"] = pd.read_csv("data/base_data_prompt.csv")[
#     "prompt_id"
# ].tolist()[::2]
# df_prompt_topic = df_prompt_topic.drop(
#     columns=[
#         "prompt_text",
#         "id",
#         "dominant_topic",
#         "topic_prob",
#         "low_confidence",
#         "topic_composition",
#     ]
# )
# df_prompt_topic["topic_label"] = df_prompt_topic["topic_label"].str.capitalize()
# df_prompt_topic = df_prompt_topic.rename(columns={"topic_label": "topic_label_prompt"})

# Main results

In [ ]:
most_sycophantic_ids = {}
least_sycophantic_ids = {}
df_least = {}
df_most = {}

In [ ]:
for m, model in enumerate(main_models):
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")
    main_models[m] = model.capitalize().replace("-instruct", "").replace("-it", "")
    model = model.capitalize().replace("-instruct", "").replace("-it", "")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df = df.loc[
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
    ]
    df["sycophancy"] = (
        df["model_answer_constrained_no_flipped_story"]
        > df["model_answer_constrained_yes_flipped_story"]
    )
    interesting_ids = list(
        df.groupby("persona_id", dropna=True)["sycophancy"]
        .mean()
        .sort_values()
        .reset_index()["persona_id"]
        .values
    )
    least_sycophantic_ids[model] = interesting_ids[:n]
    most_sycophantic_ids[model] = interesting_ids[-n:]
    del df
    print(
        f"Least sycophantic persona IDs for model {model}: ",
        least_sycophantic_ids[model],
    )
    print(
        f"Most sycophantic persona IDs for model {model}: ", most_sycophantic_ids[model]
    )
    df_least[model] = df_personas.loc[
        df_personas["uuid"].isin(least_sycophantic_ids[model])
    ]  # .merge(df_persona_topic, left_on="uuid", right_on="persona_id", how="left")
    df_most[model] = df_personas.loc[
        df_personas["uuid"].isin(most_sycophantic_ids[model])
    ]  # .merge(df_persona_topic, left_on="uuid", right_on="persona_id", how="left")
    df_least[model] = extract_demographics(df_least[model])
    df_most[model] = extract_demographics(df_most[model])

In [ ]:
plt.rcParams.update({"font.size": 22})
for col in cat_cols:
    # plotting all categorical columns
    data = {col: [], "Sycophancy": [], "model": []}
    for model in main_models:
        data[col] += df_least[model][col].tolist() + df_most[model][col].tolist()
        data["Sycophancy"] += (["Least"] * n) + (["Most"] * n)  # * 10 for topic
        data["model"] += [model] * 2 * n  # * 10 for topic

    if col == "occupation" or col == "topic_label_persona":
        ax = sns.catplot(
            pd.DataFrame(data).sort_values(by=col).sort_values(by="model"),
            kind="count",
            col="model",
            x=col,
            hue="Sycophancy",
            aspect=3,
            sharex=False,
            palette="Set2",
        )
        ax.set_xticklabels(rotation=90)

    else:
        ax = sns.catplot(
            pd.DataFrame(data).sort_values(by=col).sort_values(by="model"),
            kind="count",
            col="model",
            x=col,
            hue="Sycophancy",
            sharex=False,
            palette="Set2",
        )
        if col in [
            "race",
            "education",
            "nationality",
            "income",
            "religion",
            "community_connected",
        ]:
            ax.set_xticklabels(rotation=45)
    plt.savefig(f"figures/{col}.pdf", bbox_inches="tight")
    plt.show()

vulnerability_counter = Counter()

for col in num_cols:
    data = {col: [], "Sycophancy": [], "Model": []}
    diff = {}
    for model in main_models:
        data[col] += df_least[model][col].tolist() + df_most[model][col].tolist()
        data["Sycophancy"] += (["Least"] * n) + (["Most"] * n)  # * 10 for topic
        data["Model"] += [model] * 2 * n  # * 10 for topic
        diff[model] = (
            " *"
            if ttest_ind(
                df_least[model][col].tolist(), df_most[model][col].tolist()
            ).pvalue
            < p
            else ""
        )
        if col == "vulnerability_score":
            sorted_df = df_most[model].sort_values(col)[["uuid", col]]
            sorted_df = sorted_df.loc[sorted_df[col] > 3]
            vulnerability_counter.update(sorted_df["uuid"].tolist())

    # Only plotting numerical columns that are different between least and most sycophantic personas for at least 2 models
    if sum([d == " *" for d in diff.values()]) < 2:
        continue

    ax = sns.barplot(
        pd.DataFrame(data).sort_values(by="Model"),
        x="Model",
        y=col,
        hue="Sycophancy",
        palette="Set2",
    )

    labels = ax.get_xticklabels()
    for i, l in enumerate(labels):
        labels[i] = l.get_text() + diff[l.get_text()]
    ax.set_xticklabels(labels, rotation=90)
    plt.savefig(f"figures/{col}.pdf", bbox_inches="tight")
    plt.show()

print(vulnerability_counter)

In [ ]:
models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3-8B",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
    # "Olmo-3-7B-Instruct",
]
df_vulnerability = pd.read_csv(
    "data/personas_demographics_vulnerability_final (1).csv"
)[
    [
        "uuid",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
]
big_df = pd.DataFrame([])
for model in models:
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df = df.loc[
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
    ]
    df["sycophancy"] = (
        df["model_answer_constrained_no_flipped_story"]
        > df["model_answer_constrained_yes_flipped_story"]
    ) * 100
    df = df.groupby(["persona_id", "persona_type"])["sycophancy"].mean().reset_index()
    df["model"] = model.capitalize().replace("-instruct", "").replace("-it", "")
    df = df.merge(df_vulnerability, left_on="persona_id", right_on="uuid")
    big_df = pd.concat([df, big_df], ignore_index=True)

In [ ]:
big_df[
    [
        "sycophancy",
        "inherent_vulnerability",
        "situational_vulnerability",
        "pathogenic_vulnerability",
        "vulnerability_score",
    ]
].corr()

In [ ]:
for v_col in [
    "inherent_vulnerability",
    "situational_vulnerability",
    "pathogenic_vulnerability",
    "vulnerability_score",
]:
    ax = sns.catplot(
        big_df.sort_values(by=v_col).rename(
            columns={"sycophancy": "Sycophancy", "model": "Model"}
        ),
        kind="bar",
        col="Model",
        x=v_col,
        y="Sycophancy",
        sharex=False,
        palette="Set2",
    )
    plt.savefig(f"figures/{v_col}_syco.pdf")

# Mitigation results

In [ ]:
most_sycophantic_ids = {}
least_sycophantic_ids = {}
df_least = {}
df_most = {}

In [ ]:
for m, model in enumerate(mitigation_models):
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")
    mitigation_models[m] = model.replace("3-8B", "").replace("-3.1-8B-Instruct", "")
    model = model.replace("3-8B", "").replace("-3.1-8B-Instruct", "")
    if "dpo_model" in model:
        mitigation_models[m] = model.split("dpo_model_")[1] + " DPO"
        model = model.split("dpo_model_")[1] + " DPO"
    elif "mitigate" in model:
        mitigation_models[m] = model.replace("_mitigate", " Prompt")
        model = model.replace("_mitigate", " Prompt")
    # mitigation_models[m] = model.capitalize()
    # model = model.capitalize()

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df = df.loc[
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
    ]
    df["sycophancy"] = (
        df["model_answer_constrained_no_flipped_story"]
        > df["model_answer_constrained_yes_flipped_story"]
    )
    interesting_ids = list(
        df.groupby("persona_id", dropna=True)["sycophancy"]
        .mean()
        .sort_values()
        .reset_index()["persona_id"]
        .values
    )
    least_sycophantic_ids[model] = interesting_ids[:n]
    most_sycophantic_ids[model] = interesting_ids[-n:]
    del df
    print(
        f"Least sycophantic persona IDs for model {model}: ",
        least_sycophantic_ids[model],
    )
    print(
        f"Most sycophantic persona IDs for model {model}: ", most_sycophantic_ids[model]
    )
    df_least[model] = df_personas.loc[
        df_personas["uuid"].isin(least_sycophantic_ids[model])
    ]  # .merge(df_persona_topic, left_on="uuid", right_on="persona_id", how="left")
    df_most[model] = df_personas.loc[
        df_personas["uuid"].isin(most_sycophantic_ids[model])
    ]  # .merge(df_persona_topic, left_on="uuid", right_on="persona_id", how="left")
    df_least[model] = extract_demographics(df_least[model])
    df_most[model] = extract_demographics(df_most[model])

In [ ]:
plt.rcParams.update({"font.size": 22})
vulnerability_counter = Counter()

data = {"vulnerability_score": [], "Sycophancy": [], "model": []}
diff = {}
for model in mitigation_models:
    data["vulnerability_score"] += (
        df_least[model]["vulnerability_score"].tolist()
        + df_most[model]["vulnerability_score"].tolist()
    )
    data["Sycophancy"] += (["Least"] * n) + (["Most"] * n)  # * 10 for topic
    data["model"] += [model] * 2 * n  # * 10 for topic
    diff[model] = (
        " *"
        if ttest_ind(
            df_least[model]["vulnerability_score"].tolist(),
            df_most[model]["vulnerability_score"].tolist(),
        ).pvalue
        < (0.05 / len(mitigation_models))
        else ""
    )
    if col == "vulnerability_score":
        sorted_df = df_most[model].sort_values("vulnerability_score")[
            ["uuid", "vulnerability_score"]
        ]
        sorted_df = sorted_df.loc[sorted_df["vulnerability_score"] > 3]
        vulnerability_counter.update(sorted_df["uuid"].tolist())

# Only plotting numerical columns that are different between least and most sycophantic personas for at least 2 models
# if sum([d == " *" for d in diff.values()]) < 1:
#     continue

ax = sns.barplot(
    pd.DataFrame(data).sort_values(by="model"),
    x="model",
    y="vulnerability_score",
    hue="Sycophancy",
    palette="Set2",
)

labels = ax.get_xticklabels()
for i, l in enumerate(labels):
    labels[i] = l.get_text() + diff[l.get_text()]
ax.set_xticklabels(labels, rotation=45)
ax.set_xlabel("Model")
ax.set_ylabel("Total vulnerability score")
ax.legend(loc="best")
# ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.savefig(f"figures/vulnerability_score_mitigation.pdf", bbox_inches="tight")
plt.show()

print(vulnerability_counter)